# Web and security

A small FastAPI API, an online shop's, and what it takes to protect it:

1. store passwords as derived hashes, never in plain text;
2. sign up and log in a user;
3. accept only the webhooks signed by the payment provider;
4. encrypt a sensitive field (an IBAN) in the database;
5. (optional) hand out a session token that expires.

The API runs on no server: `TestClient` sends it the requests directly, as in a test. Each exercise creates its own application in its cell: running the cell again starts from scratch.

## Setup

Colab already has `fastapi` and `cryptography`. The next cell installs `httpx2`, the HTTP client `TestClient` uses (without it, Starlette falls back on `httpx` and prints a warning), and elsewhere (a Codespace, a local environment) whatever is missing.

In [ ]:
%pip install -q fastapi httpx2 cryptography

In [ ]:
import hashlib
import hmac
import json
import secrets
import time
from typing import Annotated

from cryptography.fernet import Fernet, InvalidToken
from fastapi import FastAPI, Header, HTTPException, Request
from fastapi.testclient import TestClient
from pydantic import BaseModel

## 1. Deriving a password

Write `derive(password, salt)`, which computes a password's hash with `hashlib.scrypt`, with one of OWASP's settings for scrypt (`n=2**14`, `r=8`, `p=5`), and `verify(password, salt, digest)`, which says whether a password gives back the stored hash.

Then check them:

- `verify` accepts the right password and refuses another one;
- two users who chose the same password, each with their own salt: are their hashes equal? Why is that intended?
- measure the duration of one derivation: how many passwords per second can an attacker who stole the database try, on this processor?

**Useful documentation**

- https://docs.python.org/3/library/time.html#time.perf_counter

In [ ]:
def derive(password: bytes, salt: bytes) -> bytes: ...


def verify(password: bytes, salt: bytes, digest: bytes) -> bool: ...


# Your code here

### Additional hints

`hashlib.scrypt(password, salt=salt, n=2**14, r=8, p=5)` returns 64 bytes; its parameters are keyword-only, except the first one. `verify` derives the proposed password again, with the user's salt, and compares the two hashes with `hmac.compare_digest`, in constant time. A salt: `secrets.token_bytes(16)`, one per user.

`verify(b"password", salt, derive(b"password", salt))` returns `True`, `verify(b"other", salt, ...)` returns `False`. For the duration: `time.perf_counter()` before and after a call to `derive`.

```python
def derive(password: bytes, salt: bytes) -> bytes:
  return hashlib.scrypt(password, salt=salt, n=..., r=..., p=...)


def verify(password: bytes, salt: bytes, digest: bytes) -> bool:
  return hmac.compare_digest(..., digest)


salt = secrets.token_bytes(16)
digest = derive(b"password", salt)
print(verify(b"password", salt, digest), verify(b"other", salt, digest))

start = time.perf_counter()
derive(b"password", salt)
duration = ...
```

### Solution

In [ ]:
def derive(password: bytes, salt: bytes) -> bytes:
  return hashlib.scrypt(password, salt=salt, n=2**14, r=8, p=5)


def verify(password: bytes, salt: bytes, digest: bytes) -> bool:
  return hmac.compare_digest(derive(password, salt), digest)


salt = secrets.token_bytes(16)
digest = derive(b"password", salt)
print(verify(b"password", salt, digest), verify(b"other", salt, digest))

# Same password, two salts: two hashes. An attacker must attack each account
# separately, and a precomputed table of hashes is of no use to them.
other_salt = secrets.token_bytes(16)
print(digest == derive(b"password", other_salt))

# A few tenths of a second per attempt, and 16 MiB of memory each (128 * n * r
# bytes): a few attempts per second and per core, against billions of SHA-256
# per second on a GPU.
start = time.perf_counter()
derive(b"password", salt)
duration = time.perf_counter() - start
print(f"{duration:.2f} s per derivation, {1 / duration:.0f} attempts per second")

## 2. Sign-up and login

Add two routes to the application, which receive a JSON body `{"username": ..., "password": ...}` (the `Credentials` class):

- `POST /signup` stores the user's salt and hash in `accounts` (never their password), and answers 201; a username already taken answers 409;
- `POST /login` answers 200 and `{"message": "Welcome, <username>"}` if the password is the right one, 401 otherwise.

Test them with `api`, the application's `TestClient`: the sign-up of `ada`, a second sign-up of `ada`, a login with the right password, then with a wrong one, then with an unknown username. What must the API answer in these last two cases, and why?

**Useful documentation**

- https://fastapi.tiangolo.com/tutorial/handling-errors/#use-httpexception

In [ ]:
class Credentials(BaseModel):
  username: str
  password: str


app = FastAPI()
api = TestClient(app)
accounts: dict[str, tuple[bytes, bytes]] = {}  # username -> (salt, digest)

# Your code here

### Additional hints

The password received is a `str`: `.encode()` gives it as bytes to `derive` and `verify`. An HTTP error is raised: `raise HTTPException(status_code=409, detail="username already taken")`. A wrong password and an unknown username get the same answer, 401: a different message would tell an attacker which accounts exist.

The test calls, and the expected codes: `api.post("/signup", json={"username": "ada", "password": "lovelace"})` → 201, then 409; the login → 200, then 401 and 401. `response.status_code` and `response.json()` give the code and the body.

```python
@app.post("/signup", status_code=201)
def sign_up(credentials: Credentials) -> dict[str, str]:
  if credentials.username in accounts:
    raise HTTPException(status_code=409, detail="username already taken")
  salt = ...
  accounts[credentials.username] = (salt, ...)
  return {"message": "account created"}


@app.post("/login")
def log_in(credentials: Credentials) -> dict[str, str]:
  if credentials.username not in accounts:
    raise HTTPException(status_code=401, detail="wrong username or password")
  salt, digest = accounts[credentials.username]
  if not verify(...):
    raise ...
  return {"message": f"Welcome, {credentials.username}"}
```

### Solution

In [ ]:
class Credentials(BaseModel):
  username: str
  password: str


app = FastAPI()
api = TestClient(app)
accounts: dict[str, tuple[bytes, bytes]] = {}  # username -> (salt, digest)


@app.post("/signup", status_code=201)
def sign_up(credentials: Credentials) -> dict[str, str]:
  if credentials.username in accounts:
    raise HTTPException(status_code=409, detail="username already taken")
  salt = secrets.token_bytes(16)
  accounts[credentials.username] = (
    salt,
    derive(credentials.password.encode(), salt),
  )
  return {"message": "account created"}


@app.post("/login")
def log_in(credentials: Credentials) -> dict[str, str]:
  # Same answer for an unknown username and a wrong password: the API doesn't
  # tell which accounts exist.
  denied = HTTPException(status_code=401, detail="wrong username or password")
  if credentials.username not in accounts:
    raise denied
  salt, digest = accounts[credentials.username]
  if not verify(credentials.password.encode(), salt, digest):
    raise denied
  return {"message": f"Welcome, {credentials.username}"}


ada = {"username": "ada", "password": "lovelace"}
for path, body in [
  ("/signup", ada),
  ("/signup", ada),
  ("/login", ada),
  ("/login", {"username": "ada", "password": "babbage"}),
  ("/login", {"username": "alan", "password": "turing"}),
]:
  response = api.post(path, json=body)
  print(path, body["username"], response.status_code, response.json())

print(accounts["ada"][1].hex()[:16], "...")  # what the database holds: the hash

## 3. Verifying a webhook

The shop's payment provider notifies it of a payment by calling `POST /webhooks/payment`, with a JSON body such as `{"order": 42, "amount": 1999}`. Anyone can call this address: to prove the call comes from it, the provider signs the request body, byte for byte, with HMAC-SHA256 and a secret that only the shop and itself know (`WEBHOOK_KEY`), and sends the signature, in hexadecimal, in the `X-Signature` header.

Write the route: it answers 401 if the signature is missing or doesn't match the body received, and otherwise adds the payment to `payments` and answers 200.

Test it as the provider would, then as an attacker: a signed body, the same body with another amount but the original signature, a body without a signature. Why does the route read the raw body rather than a Pydantic model, as in exercise 2?

**Useful documentation**

- https://fastapi.tiangolo.com/tutorial/header-params/
- https://fastapi.tiangolo.com/advanced/using-request-directly/
- https://www.starlette.dev/requests/#body

In [ ]:
WEBHOOK_KEY = b"secret-shared-with-the-provider"  # in production: outside the code

app = FastAPI()
api = TestClient(app)
payments: list[dict[str, int]] = []

# Your code here

### Additional hints

The route receives the request itself (`request: Request`) and the `X-Signature` header through a parameter `x_signature: Annotated[str, Header()] = ""`: FastAPI turns the `_` into `-`, and the default value, an empty string, avoids the 422 error when the header is missing. The raw body is obtained with `await request.body()`, hence an `async def` route. The expected signature: `hmac.new(WEBHOOK_KEY, body, hashlib.sha256).hexdigest()`, compared with `hmac.compare_digest`.

On the provider's side: `api.post("/webhooks/payment", content=body, headers={"X-Signature": signature})`, with `body = json.dumps({...}).encode()`. Expected codes: 200, 401, 401.

```python
@app.post("/webhooks/payment")
async def receive_payment(
  request: Request, x_signature: Annotated[str, Header()] = ""
) -> dict[str, str]:
  body = await request.body()
  expected = ...
  if not hmac.compare_digest(x_signature, expected):
    raise ...
  payments.append(json.loads(body))
  return {"status": "received"}


body = json.dumps({"order": 42, "amount": 1999}).encode()
signature = hmac.new(WEBHOOK_KEY, body, hashlib.sha256).hexdigest()
response = api.post("/webhooks/payment", content=body, headers={"X-Signature": ...})
```

### Solution

In [ ]:
WEBHOOK_KEY = b"secret-shared-with-the-provider"  # in production: outside the code

app = FastAPI()
api = TestClient(app)
payments: list[dict[str, int]] = []


@app.post("/webhooks/payment")
async def receive_payment(
  request: Request, x_signature: Annotated[str, Header()] = ""
) -> dict[str, str]:
  # The signature covers the bytes received: a Pydantic model parsed then
  # serialized again wouldn't necessarily give the same ones (spaces, key order).
  body = await request.body()
  expected = hmac.new(WEBHOOK_KEY, body, hashlib.sha256).hexdigest()
  if not hmac.compare_digest(x_signature, expected):
    raise HTTPException(status_code=401, detail="invalid signature")
  payments.append(json.loads(body))
  return {"status": "received"}


body = json.dumps({"order": 42, "amount": 1999}).encode()
signature = hmac.new(WEBHOOK_KEY, body, hashlib.sha256).hexdigest()
forged = json.dumps({"order": 42, "amount": 1}).encode()

for name, content, headers in [
  ("signed", body, {"X-Signature": signature}),
  ("amount changed", forged, {"X-Signature": signature}),
  ("no signature", body, {}),
]:
  response = api.post("/webhooks/payment", content=content, headers=headers)
  print(name, response.status_code, response.json())

print(payments)

## 4. Encrypting a sensitive field

The shop keeps its customers' IBAN to refund them. Write two routes:

- `POST /records` receives `{"name": ..., "iban": ...}` (the `NewRecord` class), stores in `records` the name in plain text and the IBAN encrypted with Fernet, and answers 201 with the record's number (its position in the list);
- `GET /records/{number}` returns the name and the decrypted IBAN, or 404 if the record doesn't exist.

Create a record, read it back through the API, and print what `records` holds, what someone who stole the database would see. Why is the IBAN encrypted, when the password was derived?

**Useful documentation**

- https://fastapi.tiangolo.com/tutorial/path-params/

In [ ]:
class NewRecord(BaseModel):
  name: str
  iban: str


fernet = Fernet(Fernet.generate_key())  # in production: an environment variable
app = FastAPI()
api = TestClient(app)
records: list[dict[str, str | bytes]] = []

# Your code here

### Additional hints

`fernet.encrypt` and `fernet.decrypt` take and return bytes: `.encode()` before, `.decode()` after. A path parameter is a parameter of the function with the same name as in the route (`{number}` → `number: int`).

With `{"name": "Ada Lovelace", "iban": "FR76 3000 6000 0112 3456 7890 189"}`, the creation answers 201 and `{"number": 0}`, reading `/records/0` gives back the IBAN, reading `/records/5` answers 404. In `records`, the IBAN is a token starting with `gAAAAA`.

```python
@app.post("/records", status_code=201)
def create_record(record: NewRecord) -> dict[str, int]:
  records.append({"name": record.name, "iban": fernet.encrypt(...)})
  return {"number": ...}


@app.get("/records/{number}")
def read_record(number: int) -> dict[str, str]:
  if not 0 <= number < len(records):
    raise ...
  record = records[number]
  return {"name": record["name"], "iban": ...}
```

### Solution

In [ ]:
class NewRecord(BaseModel):
  name: str
  iban: str


fernet = Fernet(Fernet.generate_key())  # in production: an environment variable
app = FastAPI()
api = TestClient(app)
records: list[dict[str, str | bytes]] = []


@app.post("/records", status_code=201)
def create_record(record: NewRecord) -> dict[str, int]:
  records.append({"name": record.name, "iban": fernet.encrypt(record.iban.encode())})
  return {"number": len(records) - 1}


@app.get("/records/{number}")
def read_record(number: int) -> dict[str, str]:
  if not 0 <= number < len(records):
    raise HTTPException(status_code=404, detail="record not found")
  record = records[number]
  return {"name": record["name"], "iban": fernet.decrypt(record["iban"]).decode()}


response = api.post(
  "/records", json={"name": "Ada Lovelace", "iban": "FR76 3000 6000 0112 3456 7890 189"}
)
print(response.status_code, response.json())
print(api.get("/records/0").json())
print(api.get("/records/5").status_code)

# In the database: a token unreadable without the key, which stays outside the database.
# Encrypt, because the shop must read the IBAN back to refund; a password only has
# to be compared, and a derivation can't be undone, even with a key.
print(records[0])

## 5. (Optional) A session token that expires

Rather than sending their password again with each request, the logged-in user receives a session token: the username encrypted with Fernet, which also carries its creation time. Reuse `accounts` and `verify` from exercises 1 and 2 (where `ada` is signed up):

- `POST /login` checks the password, as in exercise 2, and answers `{"token": ...}`;
- `GET /profile` reads the token in the `Authorization: Bearer <token>` header, and answers `{"username": ...}` if the token is valid and less than an hour old, 401 otherwise.

Test with a token just obtained, a modified token, and a token created two hours ago, without waiting two hours.

**Useful documentation**

- https://cryptography.io/en/latest/fernet/#cryptography.fernet.Fernet.encrypt_at_time

In [ ]:
sessions = Fernet(Fernet.generate_key())  # a separate key, for the tokens only
app = FastAPI()
api = TestClient(app)

# Your code here

### Additional hints

`sessions.decrypt(token.encode(), ttl=3600)` raises `InvalidToken` for a token that was modified, comes from another key or is more than an hour old: catch it to answer 401. The header: `authorization: Annotated[str, Header()] = ""`, then `authorization.removeprefix("Bearer ")`. A token from two hours ago: `sessions.encrypt_at_time(b"ada", int(time.time()) - 7200)`.

Expected codes: 200 and `{"username": "ada"}`, then 401 and 401.

```python
@app.post("/login")
def log_in(credentials: Credentials) -> dict[str, str]:
  ...  # the check of exercise 2
  return {"token": sessions.encrypt(...).decode()}


@app.get("/profile")
def profile(authorization: Annotated[str, Header()] = "") -> dict[str, str]:
  token = authorization.removeprefix("Bearer ")
  try:
    username = ...
  except InvalidToken:
    raise ... from None
  return {"username": username}


token = api.post("/login", json=...).json()["token"]
response = api.get("/profile", headers={"Authorization": f"Bearer {token}"})
```

### Solution

In [ ]:
sessions = Fernet(Fernet.generate_key())  # a separate key, for the tokens only
app = FastAPI()
api = TestClient(app)


@app.post("/login")
def log_in(credentials: Credentials) -> dict[str, str]:
  denied = HTTPException(status_code=401, detail="wrong username or password")
  if credentials.username not in accounts:
    raise denied
  salt, digest = accounts[credentials.username]
  if not verify(credentials.password.encode(), salt, digest):
    raise denied
  return {"token": sessions.encrypt(credentials.username.encode()).decode()}


@app.get("/profile")
def profile(authorization: Annotated[str, Header()] = "") -> dict[str, str]:
  token = authorization.removeprefix("Bearer ")
  try:
    username = sessions.decrypt(token.encode(), ttl=3600).decode()
  except InvalidToken:
    raise HTTPException(status_code=401, detail="invalid or expired token") from None
  return {"username": username}


token = api.post("/login", json={"username": "ada", "password": "lovelace"}).json()[
  "token"
]
modified = token[:-6] + ("A" if token[-6] != "A" else "B") + token[-5:]
old = sessions.encrypt_at_time(b"ada", int(time.time()) - 7200).decode()

for name, value in [("valid", token), ("modified", modified), ("old", old)]:
  response = api.get("/profile", headers={"Authorization": f"Bearer {value}"})
  print(name, response.status_code, response.json())